# Gemini book embeddings setup

Prepare the database connection and inspect the book and embedding tables before implementing embedding ingestion. This notebook currently checks setup and schema; it does not generate or store embeddings.

## Current implementation scope

The notebook now generates and stores embeddings for a 10-book sample, then checks the inserted records.

## Resumable ingestion

The later steps process books not yet embedded for the selected model, save vectors in batches, and retry rate-limited requests.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from sqlalchemy import create_engine, text, inspect

SUPABASE_DATABASE_URL = os.getenv("SUPABASE_DATABASE_URL")
engine = create_engine(SUPABASE_DATABASE_URL)

## Check database connectivity

Run a lightweight query to confirm the database connection is available.

In [ ]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Connection successful:", result.scalar())

except Exception as e:
    print("Connection failed:", e)

## Register the pgvector type

Teach SQLAlchemy to recognize PostgreSQL `vector` columns so their schema can be inspected correctly.

In [ ]:
from pgvector.sqlalchemy import VECTOR
from sqlalchemy.dialects.postgresql.base import ischema_names

ischema_names["vector"] = VECTOR

## Inspect the destination tables

Review the columns, nullability, and foreign keys for `books` and `book_embeddings` before adding embedding records.

In [ ]:
inspector = inspect(engine)

for table in ["books", "book_embeddings"]:
    print(f"\n{table.upper()}")

    for column in inspector.get_columns(table):
        print(
            column["name"],
            column["type"],
            "nullable:", column["nullable"]
        )

    print("Foreign keys:", inspector.get_foreign_keys(table))

In [ ]:
import pandas as pd

## Load book text for processing

Fetch each book ID, title, and description from the database as input for the embedding workflow.

In [ ]:
with engine.connect() as conn:
    df_books = pd.read_sql(
        text("SELECT id, title, description FROM books"),
        conn
    )

df_books.sort_values("id").reset_index(drop=True)

## Check for repeated descriptions

Find books that share an identical description so repeated text can be reviewed before embedding.

In [ ]:
duplicates = df_books[
    df_books.duplicated(subset=["description"], keep=False)
].sort_values("description")

print(f"Books with duplicated descriptions: {len(duplicates)}")
print(f"Unique duplicated descriptions: {duplicates['description'].nunique()}")

duplicates

## Exclude boilerplate descriptions

Filter out descriptions containing the known reproduction notice to create the pending set for later processing.

In [ ]:
boilerplate = "This is a reproduction of the original artefact."

df_pending = df_books[
    ~df_books["description"].str.contains(
        boilerplate,
        case=False,
        na=False,
        regex=False
    )
].copy()

df_pending.sort_values("id").reset_index(drop=True)

## Build embedding documents

Wrap each pending book description as a LangChain `Document` and attach its book ID so a future embedding can be linked back to its source.

In [ ]:
from langchain_core.documents import Document

In [ ]:
documents = [
    Document(
        page_content=row.description,
        metadata={"book_id": int(row.id)}
    )
    for row in df_pending.itertuples(index=False)
]

print(f"Documents created: {len(documents)}")

In [ ]:
documents[0]

## Check Gemini credentials

Confirm the API key is configured before making a request to Gemini.

In [ ]:
assert os.getenv("GOOGLE_API_KEY"), "Missing GOOGLE_API_KEY"

## Configure the embedding model

Set up Gemini to embed book descriptions for document retrieval, with vectors sized to match the 768-dimension database column.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

MODEL_NAME = os.getenv("GEMINI_EMBEDDING_MODEL")
EMBEDDING_DIMENSIONS = 768

assert MODEL_NAME, "Missing GEMINI_EMBEDDING_MODEL"

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODEL_NAME,
    task_type="RETRIEVAL_DOCUMENT",
    output_dimensionality=EMBEDDING_DIMENSIONS,
)

## Generate a sample embedding

Embed one book description and inspect its source ID and vector dimensions before processing the full set.

In [ ]:
test_vector = embeddings.embed_documents(
    [documents[0].page_content]
)[0]

print("Book ID:", documents[0].metadata["book_id"])
print("Vector dimensions:", len(test_vector))
print("First 5 values:", test_vector[:5])

In [ ]:
for column in inspector.get_columns("book_embeddings"):
    print(
        column["name"],
        column["type"],
        "nullable:", column["nullable"],
        "default:", column["default"]
    )

## Reflect the embedding table

Load the existing table definition so inserts use the database column types and constraints.

In [ ]:
from sqlalchemy import MetaData, Table

metadata = MetaData()

book_embeddings = Table(
    "book_embeddings",
    metadata,
    autoload_with=engine
)

## Skip completed books

Load book IDs already embedded for this model so reruns process only books that remain.

In [ ]:
with engine.connect() as conn:
    df_existing_embeddings = pd.read_sql(
        text("""
            SELECT book_id
            FROM book_embeddings
            WHERE model = :model
        """),
        conn,
        params={"model": MODEL_NAME}
    )

existing_book_ids = set(df_existing_embeddings["book_id"])

print(f"Already embedded for {MODEL_NAME}: {len(existing_book_ids)}")

In [ ]:
df_remaining = df_pending[
    ~df_pending["id"].isin(existing_book_ids)
].copy()

print(f"Remaining books to embed: {len(df_remaining)}")
df_remaining.head()

In [ ]:
remaining_documents = [
    Document(
        page_content=row.description,
        metadata={"book_id": int(row.id)}
    )
    for row in df_remaining.itertuples(index=False)
]

print(f"Remaining documents: {len(remaining_documents)}")

## Limit each daily ingestion run

Process up to 800 books per day against the 1,000-call Gemini daily quota, leaving about 200 calls for app testing. Run the ingestion daily; the existing-book check lets it resume with books not yet embedded.

In [ ]:
MAX_BOOKS_PER_RUN = 800

documents_to_process = remaining_documents[:MAX_BOOKS_PER_RUN]

print(f"Books remaining: {len(remaining_documents)}")
print(f"Books selected for this run: {len(documents_to_process)}")

## Split documents into batches

Divide remaining documents into manageable groups before sending them to the embedding API.

In [ ]:
def chunk_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

## Generate and store remaining embeddings

⚠️ Run the preceding cells first to refresh `documents_to_process` for the selected model and exclude books that already have embeddings, avoiding duplicate inserts.

Embed each batch, store its vectors with the source book IDs in a transaction, and retry rate-limited requests with backoff. A short delay separates batches.

In [ ]:
import time

BATCH_SIZE = 25
BATCH_DELAY = 20
MAX_RETRIES = 2

total_inserted = 0

for batch_number, batch_docs in enumerate(
    chunk_list(documents_to_process, BATCH_SIZE), start=1
):
    texts = [doc.page_content for doc in batch_docs]

    for attempt in range(MAX_RETRIES):
        try:
            vectors = embeddings.embed_documents(texts)

            records = [
                {
                    "book_id": doc.metadata["book_id"],
                    "model": MODEL_NAME,
                    "dimensions": EMBEDDING_DIMENSIONS,
                    "embedding": vector
                }
                for doc, vector in zip(batch_docs, vectors, strict=True)
            ]

            with engine.begin() as conn:
                conn.execute(book_embeddings.insert(), records)

            total_inserted += len(records)

            print(f"Batch {batch_number}: {len(records)} inserted")
            break

        except Exception as e:
            if "RESOURCE_EXHAUSTED" not in str(e) and "429" not in str(e):
                raise

            if attempt == MAX_RETRIES - 1:
                print("Maximum retries reached. Stopping ingestion.")
                raise

            delay = 30 * (2 ** attempt)
            print(f"Rate limit reached. Retrying after {delay}s...")
            time.sleep(delay)

    time.sleep(BATCH_DELAY)

print(f"Total inserted: {total_inserted}")

## Configure query embeddings

Use the same Gemini model and vector size with the `RETRIEVAL_QUERY` task for search text.

In [ ]:
query_embeddings = GoogleGenerativeAIEmbeddings(
    model=MODEL_NAME,
    task_type="RETRIEVAL_QUERY",
    output_dimensionality=EMBEDDING_DIMENSIONS,
)

## Embed a natural-language query

Convert the sample search text into a vector that can be compared with stored book embeddings.

In [ ]:
query = "A story about friendship, adventure, and discovering new places"

query_vector = query_embeddings.embed_query(query)

print(f"Query embedding dimensions: {len(query_vector)}")

## Retrieve similar books

Use pgvector cosine distance to rank books for the query vector, restricted to the selected model, and return the five closest matches.

In [ ]:
with engine.connect() as conn:
    results = pd.read_sql(
        text("""
            SELECT
                b.id,
                b.title,
                b.description,
                1 - (e.embedding <=> CAST(:query_vector AS vector)) AS similarity
            FROM book_embeddings e
            JOIN books b ON b.id = e.book_id
            WHERE e.model = :model
            ORDER BY e.embedding <=> CAST(:query_vector AS vector)
            LIMIT 5
        """),
        conn,
        params={
            "query_vector": str(query_vector),
            "model": MODEL_NAME
        }
    )

In [ ]:
results